# 04. Torneio de Algoritmos em Validação Walk-Forward e Ablation Study
**Artefato Analítico-Preditivo para Planejamento de Demanda e Mitigação de Desperdício**  
*Mestrado Profissional em Educação Profissional e Tecnológica (ProfEPT) / Gestão Pública*  
**Pesquisador:** Michel Silva de Souza  

---

## Storytelling: O Confronto de Inteligência e a Taxonomia de Ensembles
Com a garantia matemática de isolamento temporal estrito (Zero Data Leakage), submetemos os dados a um **torneio rigoroso de Machine Learning** avaliado em validação *Walk-Forward* ao longo de 2026 (com janela móvel de 12 meses de treino anterior a cada emissão).

### Os Dois Horizontes Decisórios Reais:
1. **$T+1$ (Preparo da Semana Seguinte - 3 a 7 dias de antecedência):** A decisão operacional que pode incorporar os cardápios publicados e a estimativa técnica da nutricionista.
2. **$T+2$ (Compras Quinzenais de Suprimentos - 10 a 14 dias de antecedência):** Decisão estritamente autônoma, na qual não há reservas discentes nem estimativas humanas disponíveis.

### A Taxonomia Completa de Ensembles Testada:
* **Bagging (Bootstrap Aggregating):** Random Forest e ExtraTrees (árvores paralelas com amostragem bootstrap independente para redução máxima de variância).
* **Boosting:** Gradient Boosting, AdaBoost, LightGBM, XGBoost e CatBoost (aprendizado sequencial nos resíduos ponderados).
* **Voting (Votação):** Média ponderada de estimadores heterogêneos (Ridge + ExtraTrees + CatBoost).
* **Stacking (Stacked Generalization):** Meta-aprendizado combinando modelos heterogêneos de primeiro nível através de um meta-regressor regularizado.
* **Blending:** Divisão temporal com holdout recente sem vazamento e cálculo de pesos não-negativos via NNLS.
* **Cascading (Cascata):** Modelo em dois estágios (estágio 1: âncora de tendência global; estágio 2: modelagem dos resíduos não-lineares).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.core.config import RESULTADOS
from src.core.models import PASTA_MODELOS

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['figure.dpi'] = 130


### 1. Carregamento dos Resultados Consolidados do Torneio
Matriz de métricas calculada para todos os modelos, cenários de informação e horizontes.

In [ ]:
df_metricas = pd.read_csv(PASTA_MODELOS / "metricas_torneio_modelos.csv", sep=";")
print(f"Total de configurações avaliadas: {len(df_metricas)}")
df_metricas.head(10)


## 2. Horizonte T+1 (Preparo Diário): O Impacto dos Cenários e dos Ensembles
No horizonte de curto prazo ($T+1$), avaliamos como o acréscimo de Cardápio, Clima e Nutricionista reduz o erro médio (MAE) e o WAPE.

In [ ]:
t1_df = df_metricas[df_metricas["horizonte"] == "t1"].sort_values("MAE")
t1_top = t1_df[["cenario", "algoritmo", "tipo_ensemble", "MAE", "RMSE", "WAPE_pct", "Total_Sobra_ref", "Total_Deficit_ref", "Taxa_Dias_Deficit_pct"]].head(12)
t1_top


### Visualização: Comparativo de MAE por Família de Ensemble em T+1

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
c4_df = t1_df[t1_df["cenario"] == "Cenario_4_Com_Nutricionista"].sort_values("MAE", ascending=False)

cores = ["#27ae60" if "Ensemble" in a else "#3498db" if "Ridge" in a or "Linear" in a else "#7f8c8d" for a in c4_df["algoritmo"]]
ax.barh(c4_df["algoritmo"], c4_df["MAE"], color=cores, height=0.65)
ax.set_title("Desempenho dos Algoritmos e Ensembles no Horizonte T+1 (Cenário Completo)", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("MAE (Erro Médio Absoluto em Refeições)")
for i, (mae, alg) in enumerate(zip(c4_df["MAE"], c4_df["algoritmo"])):
    ax.text(mae + 0.3, i, f"{mae:.2f} ref", va="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()


## 3. Horizonte T+2 (Compras Quinzenais): O Campeão Autônomo
Para a gestão de compras com 10 a 14 dias de antecedência, nenhum dado humano pode ser utilizado. Identificamos os modelos que alcançam o menor desperdício e erro médio de forma 100% independente.

In [ ]:
t2_df = df_metricas[df_metricas["horizonte"] == "t2"].sort_values("MAE")
t2_top = t2_df[["cenario", "algoritmo", "tipo_ensemble", "MAE", "RMSE", "WAPE_pct", "Total_Sobra_ref", "Total_Deficit_ref", "Taxa_Dias_Deficit_pct"]].head(10)
t2_top


### Visualização: Ranking Top 10 em Compras (T+2 Autônomo)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
t2_top_plot = t2_top.head(8).sort_values("MAE", ascending=False)

ax.barh(t2_top_plot["algoritmo"], t2_top_plot["MAE"], color="#2980b9", height=0.6)
ax.set_title("Top Algoritmos para Compras de Suprimentos (Horizonte T+2 Autônomo)", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("MAE (Refeições)")
for i, mae in enumerate(t2_top_plot["MAE"]):
    ax.text(mae + 0.3, i, f"{mae:.2f} ref", va="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()


## 4. Comparativo Sistemático das 6 Famílias de Ensemble
Resumo comparativo do comportamento de cada tipo de ensemble na modelagem preditiva da escola pública.

In [ ]:
comp_familias = df_metricas.groupby(["tipo_ensemble"])[["MAE", "RMSE", "WAPE_pct"]].mean().sort_values("MAE").round(2)
comp_familias.columns = ["MAE Médio", "RMSE Médio", "WAPE Médio (%)"]
comp_familias


## Conclusões do Torneio para a Dissertação

1. **Eficiência Superior dos Ensembles de Bagging e Boosting:** Modelos como ExtraTrees, CatBoost e Random Forest superam consistentemente os modelos lineares isolados nos cenários autônomos de compras ($T+2$), capturando interações não-lineares complexas entre dias da semana e sazonalidade de término de semestre.
2. **Robustez dos Ensembles de Voting e Stacking:** A combinação de estimadores lineares regularizados (Ridge) com árvores aleatórias gera uma fronteira de predição equilibrada, reduzindo a variância e mitigando picos de erro.
3. **O Papel do Horizonte de Compras ($T+2$):** Com antecedência de duas semanas, o modelo autônomo baseado em sazonalidade e cardápio atinge excelente estabilidade operacional sem demandar tempo de preenchimento manual da equipe de nutrição.
4. **Fundamentação para o Modelo Híbrido Final:** Em $T+1$, o próximo marco aperfeiçoará o campeão combinando-o de forma adaptativa com o quantitativo da nutricionista e aplicando **Margens Conformal de Proteção Alimentar**.
